<a href="https://colab.research.google.com/github/swethas-22/Titanic-Survival-Predictor/blob/main/model_joblib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import joblib
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# 1. Load and clean the Titanic data
df = sns.load_dataset("titanic")[["survived", "pclass", "sex", "age", "fare"]].dropna()
df["sex"] = (df["sex"] == "female").astype(int)

# 2. Features and target
FEATURES = ["pclass", "sex", "age", "fare"]
X = df[FEATURES]
y = df["survived"]

# 3. Train model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

# 4. Save model to disk
joblib.dump(model, "model.joblib")
print("✅ model.joblib successfully saved!")

✅ model.joblib successfully saved!


In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# 1. Page Configuration
st.set_page_config(page_title="Titanic Survival Predictor", page_icon="🚢", layout="centered")

# Features list matching the trained model's exact order
FEATURES = ["pclass", "sex", "age", "fare"]

# 2. Cached model loading
@st.cache_resource
def load_model():
    return joblib.load("model.joblib")

# 3. User Interface Headings
st.title("🚢 Would this passenger have survived?")
st.write("An educational interactive model trained on historical Titanic passenger records.")

# 4. Input Controls
pclass = st.selectbox("Ticket Class (1 = 1st, 2 = 2nd, 3 = 3rd)", [1, 2, 3], index=2)
sex = st.radio("Sex", ["female", "male"], horizontal=True)
age = st.slider("Age (in years)", min_value=1, max_value=80, value=30, step=1)
fare = st.slider("Fare Paid (USD)", min_value=0, max_value=500, value=32, step=1)

# 5. Prediction Execution on Button Click
if st.button("Predict Survival Chance", type="primary"):
    try:
        model = load_model()
    except FileNotFoundError:
        st.error("⚠️ The model file (`model.joblib`) is missing. Train and save it first.")
        st.stop()
    except Exception as e:
        st.error(f"⚠️ Error loading model: {e}")
        st.stop()

    # Encode inputs to match the model training format
    encoded_sex = 1 if sex == "female" else 0
    row = pd.DataFrame([[pclass, encoded_sex, age, fare]], columns=FEATURES)

    # Calculate survival probability
    try:
        probabilities = model.predict_proba(row)[0]
        survival_chance = probabilities[1]

        st.markdown("---")
        st.metric(
            label="Estimated Chance of Survival",
            value=f"{survival_chance:.1%}"
        )

        if survival_chance >= 0.5:
            st.success("The model predicts this passenger likely **survived**.")
        else:
            st.info("The model predicts this passenger likely **did not survive**.")
    except Exception as e:
        st.error(f"Prediction error: {e}")

# 6. Educational Caption
st.caption("ℹ️ A learning exercise. It describes a historical dataset and is not advice.")

Writing app.py


In [4]:
%%writefile requirements.txt
streamlit
scikit-learn
pandas
joblib

Writing requirements.txt
